In [ ]:
import os
import warnings
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from torch.autograd import grad as autograd_grad

warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")

# ── Domain ────────────────────────────────────────────────────
X_MAX       = 20.0
Z_MAX       = 10.0
T_MAX       = 24.0
SLOPE_ANGLE = 30.0
RHO_W       = 1000.0
G_ACC       = 9.81
E_REF       = 1.0e6

# ── CSV Column Mappings ───────────────────────────────────────
CSV_FILE    = "/content/label_109_EMA_optimized_PINN.csv"

COL_TIME      = "timestamp"
COL_GEO       = "geo"
COL_DEPTH     = "temp"
COL_RAINFALL  = "rain"
COL_SOIL      = "soil"
RAINFALL_UNIT = "mm/hr"

SINGLE_DEPTH_SENSOR = False
DEPTH_POS_DOWN      = True
GEO_IS_LATLON       = False

TIMESTAMP_FORMATS = [
    "%m/%d/%Y %H:%M",
    "%d/%m/%Y %H:%M", "%d/%m/%Y %H:%M:%S", "%d-%m-%Y %H:%M",
    "%Y-%m-%d %H:%M", "%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M:%S",
]

# ── Soil layers (initial / prior values for inverse modeling) ─
SOIL = {
    1: dict(z_lo=8.0,  z_hi=10.0,
            theta_r=0.065, theta_s=0.410, alpha=0.75, n=1.89,
            Ks=1.22e-6,  E=5.0e6,  nu=0.30, rho_b=1600.0,
            c_prime=10.0e3, phi_prime=30.0),
    2: dict(z_lo=4.0,  z_hi=8.0,
            theta_r=0.078, theta_s=0.430, alpha=0.36, n=1.56,
            Ks=2.89e-7,  E=8.0e6,  nu=0.35, rho_b=1700.0,
            c_prime=15.0e3, phi_prime=25.0),
    3: dict(z_lo=0.0,  z_hi=4.0,
            theta_r=0.100, theta_s=0.460, alpha=0.15, n=1.25,
            Ks=5.56e-8,  E=15.0e6, nu=0.40, rho_b=1800.0,
            c_prime=20.0e3, phi_prime=20.0),
}

THETA_CLIP_LO = SOIL[3]["theta_r"]
THETA_CLIP_HI = SOIL[3]["theta_s"]

# ── Physical bounds for inverse parameters ────────────────────
# Used for clamping after each step to keep parameters physical
PARAM_BOUNDS = {
    #              L1 (Sandy CL)     L2 (Clay Loam)     L3 (Clay)
    "alpha":   [(0.01, 3.0),        (0.01, 2.0),        (0.01, 1.0)],
    "n":       [(1.10, 3.0),        (1.05, 2.5),        (1.01, 2.0)],
    "Ks":      [(1e-8, 1e-4),       (1e-9, 1e-5),       (1e-10, 1e-6)],
    "theta_r": [(0.01, 0.15),       (0.01, 0.15),       (0.05, 0.20)],
    "theta_s": [(0.30, 0.60),       (0.30, 0.60),       (0.30, 0.60)],
}


def _lp_fixed(z_norm, key):
    """Original fixed lookup — used as fallback / reference."""
    z_m = z_norm * Z_MAX
    v1 = float(SOIL[1][key]); v2 = float(SOIL[2][key]); v3 = float(SOIL[3][key])
    return torch.where(z_m >= 8.0, v1 * torch.ones_like(z_m),
           torch.where(z_m >= 4.0, v2 * torch.ones_like(z_m),
                                   v3 * torch.ones_like(z_m)))


def _lp_trainable(z_norm, key, params):
    """
    Trainable lookup — returns per-point parameter values using
    the learnable per-layer tensors stored in `params` dict.
    params[key] = 1-D tensor of shape [3] → [layer1, layer2, layer3]
    """
    z_m = z_norm * Z_MAX
    v1 = params[key][0]   # layer 1 (Sandy CL, z ≥ 8m)
    v2 = params[key][1]   # layer 2 (Clay Loam, 4–8m)
    v3 = params[key][2]   # layer 3 (Clay, 0–4m)
    return torch.where(z_m >= 8.0, v1.expand_as(z_m),
           torch.where(z_m >= 4.0, v2.expand_as(z_m),
                                   v3.expand_as(z_m)))


# ── Van Genuchten (accepts optional trainable params) ─────────
def _lp(z, key, params=None):
    """Unified dispatcher: trainable if params provided, else fixed."""
    if params is not None:
        return _lp_trainable(z, key, params)
    return _lp_fixed(z, key)


def vg_Se(psi, z, params=None):
    alpha = _lp(z, "alpha", params)
    n     = _lp(z, "n",     params)
    m     = 1.0 - 1.0 / n
    arg   = torch.clamp(alpha * torch.abs(psi), min=0.0)
    Se    = 1.0 / (1.0 + arg.pow(n)).pow(m)
    Se    = torch.where(psi >= 0.0, torch.ones_like(psi), Se)
    return torch.clamp(Se, min=1e-6, max=1.0 - 1e-6)


def vg_theta(psi, z, params=None):
    return (_lp(z, "theta_r", params)
            + vg_Se(psi, z, params) * (_lp(z, "theta_s", params)
                                       - _lp(z, "theta_r", params)))


def vg_K(psi, z, params=None):
    Ks = _lp(z, "Ks", params)
    n  = _lp(z, "n",  params)
    m  = 1.0 - 1.0 / n
    Se = vg_Se(psi, z, params)
    Se_c  = torch.clamp(Se, min=1e-6, max=1.0 - 1e-6)
    inner = torch.clamp(1.0 - Se_c.pow(1.0 / m), min=0.0)
    K     = Ks * Se.pow(0.5) * (1.0 - inner.pow(m)).pow(2.0)
    return torch.clamp(K, min=1e-15, max=1e-2)


def dtheta_dpsi(psi, z, params=None):
    alpha = _lp(z, "alpha", params)
    n     = _lp(z, "n",     params)
    m     = 1.0 - 1.0 / n
    dts   = _lp(z, "theta_s", params) - _lp(z, "theta_r", params)
    arg   = torch.clamp(alpha * torch.abs(psi), min=0.0)
    denom = 1.0 + arg.pow(n)
    C     = dts * m * n * alpha * arg.pow(n - 1.0) / denom.pow(m + 1.0)
    C     = torch.where(psi >= 0.0, torch.zeros_like(psi), C)
    return torch.clamp(C, min=0.0, max=10.0)


def inv_vg_psi(theta, z, params=None):
    alpha   = _lp(z, "alpha",   params)
    n       = _lp(z, "n",       params)
    m       = 1.0 - 1.0 / n
    theta_r = _lp(z, "theta_r", params)
    theta_s = _lp(z, "theta_s", params)
    Se      = torch.clamp((theta - theta_r) / (theta_s - theta_r),
                          min=1e-6, max=1.0 - 1e-6)
    inner   = torch.clamp(Se.pow(-1.0 / m) - 1.0, min=0.0)
    psi     = -(1.0 / alpha) * inner.pow(1.0 / n)
    return torch.clamp(psi, min=-500.0, max=0.0)


# ── Elasticity (unchanged) ────────────────────────────────────
def lame(z):
    E  = _lp(z, "E");  nu = _lp(z, "nu")
    lam = E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))
    mu  = E / (2.0 * (1.0 + nu))
    return lam, mu


def _grad(y, v):
    return autograd_grad(y, v, grad_outputs=torch.ones_like(y),
                         create_graph=True, retain_graph=True)[0]


def stress_2d(u, w, x, z):
    eps_xx = _grad(u, x); eps_zz = _grad(w, z)
    gamma  = _grad(u, z) + _grad(w, x)
    lam_t, mu_t = lame(z)
    dil    = eps_xx + eps_zz
    sxx    = (lam_t * dil + 2.0 * mu_t * eps_xx) / E_REF
    szz    = (lam_t * dil + 2.0 * mu_t * eps_zz) / E_REF
    txz    = (mu_t * gamma)                       / E_REF
    return sxx, szz, txz


# ── Neural Network — with Inverse Parameters ──────────────────
class PINN(nn.Module):
    """
    4 outputs:
      psi         — pressure head (m)
      u, w        — displacements (m)
      theta_direct— volumetric water content ∈ (θ_r, θ_s) per layer

    ── INVERSE MODELING ──────────────────────────────────────
    Hydraulic parameters (alpha, n, Ks, theta_r, theta_s) are stored
    as nn.Parameter tensors of shape [3] (one value per soil layer).
    They are optimized jointly with the network weights.

    Initial values come from SOIL dict (PTF estimates) — the optimizer
    refines them to best fit the observed sensor data while satisfying
    the Richards equation physics.
    """
    def __init__(self, hidden=6, width=128):
        super().__init__()

        # ── Network trunk ──────────────────────────────────────
        layers = [nn.Linear(3, width), nn.Tanh()]
        for _ in range(hidden - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        self.trunk      = nn.Sequential(*layers)
        self.head_psi   = nn.Linear(width, 1)
        self.head_disp  = nn.Linear(width, 2)
        self.head_theta = nn.Linear(width, 1)
        self._init_weights()

        # ── Trainable hydraulic parameters ─────────────────────
        # Shape: [3] → [layer1 (Sandy CL), layer2 (Clay Loam), layer3 (Clay)]
        # Initialized from SOIL dict (PTF priors)
        def _init_param(key):
            vals = [SOIL[i][key] for i in [1, 2, 3]]
            return nn.Parameter(torch.tensor(vals, dtype=torch.float32))

        # We train log-space for alpha, Ks (always positive, wide range)
        # and direct space for n, theta_r, theta_s (bounded, easier to constrain)
        self._log_alpha  = nn.Parameter(torch.log(_init_param("alpha").data))
        self._log_Ks     = nn.Parameter(torch.log(_init_param("Ks").data))
        self._raw_n      = nn.Parameter(_init_param("n").data - 1.0)  # n > 1
        self._raw_theta_r = nn.Parameter(_init_param("theta_r").data)
        self._raw_theta_s = nn.Parameter(_init_param("theta_s").data)

        print("\n[Inverse PINN] Trainable hydraulic parameters initialised:")
        print(f"  alpha  : {[SOIL[i]['alpha']  for i in [1,2,3]]}")
        print(f"  n      : {[SOIL[i]['n']      for i in [1,2,3]]}")
        print(f"  Ks     : {[SOIL[i]['Ks']     for i in [1,2,3]]}")
        print(f"  theta_r: {[SOIL[i]['theta_r'] for i in [1,2,3]]}")
        print(f"  theta_s: {[SOIL[i]['theta_s'] for i in [1,2,3]]}\n")

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=1.0)
                nn.init.zeros_(m.bias)

    @property
    def hydro_params(self):
        """
        Returns dict of constrained hydraulic parameters.
        Constraints ensure physical validity:
          alpha > 0  (log-space)
          n > 1      (raw + 1 + softplus for stability)
          Ks > 0     (log-space)
          theta_r > 0, theta_s > theta_r (clamped)
        """
        alpha   = torch.exp(self._log_alpha)
        Ks      = torch.exp(self._log_Ks)
        n       = 1.0 + torch.nn.functional.softplus(self._raw_n)
        theta_r = torch.clamp(self._raw_theta_r, min=0.01, max=0.25)
        # torch.clamp requires scalar bounds — enforce theta_s > theta_r via torch.max
        theta_s = torch.clamp(self._raw_theta_s, min=0.10, max=0.65)
        theta_s = torch.max(theta_s, theta_r.detach() + 0.05)
        return {
            "alpha": alpha, "n": n, "Ks": Ks,
            "theta_r": theta_r, "theta_s": theta_s,
        }

    def clamp_params_(self):
        """
        Hard clamp after each optimiser step to keep parameters
        strictly inside physical bounds (called from training loop).
        """
        with torch.no_grad():
            bounds = PARAM_BOUNDS
            for li, layer_idx in enumerate([1, 2, 3]):
                lo, hi = bounds["alpha"][li]
                self._log_alpha.data[li].clamp_(np.log(lo), np.log(hi))
                lo, hi = bounds["Ks"][li]
                self._log_Ks.data[li].clamp_(np.log(lo), np.log(hi))
                lo, hi = bounds["n"][li]
                self._raw_n.data[li].clamp_(lo - 1.0, hi - 1.0)
                lo, hi = bounds["theta_r"][li]
                self._raw_theta_r.data[li].clamp_(lo, hi)
                lo, hi = bounds["theta_s"][li]
                self._raw_theta_s.data[li].clamp_(lo, hi)

    def get_learned_params(self):
        """Returns dict of learned parameter values as numpy arrays — for logging."""
        p = self.hydro_params
        return {k: v.detach().cpu().numpy() for k, v in p.items()}

    def forward(self, x, z, t):
        feat = self.trunk(torch.cat([x, z, t], dim=1))

        # Layer-aware psi scaling
        z_m     = z * Z_MAX
        raw_psi = self.head_psi(feat)
        scale   = torch.where(z_m >= 8.0,  3.0 * torch.ones_like(z_m),
                  torch.where(z_m >= 4.0,  7.0 * torch.ones_like(z_m),
                                           25.0 * torch.ones_like(z_m)))
        shift   = torch.where(z_m >= 8.0, -1.5 * torch.ones_like(z_m),
                  torch.where(z_m >= 4.0, -4.0 * torch.ones_like(z_m),
                                          -20.0 * torch.ones_like(z_m)))
        psi = scale * torch.tanh(raw_psi) + shift

        raw_d = self.head_disp(feat)
        u     = raw_d[:, 0:1] * 1e-3
        w     = raw_d[:, 1:2] * 1e-3

        # Direct theta — uses trainable theta_r, theta_s
        p       = self.hydro_params
        raw_th  = self.head_theta(feat)
        theta_r = _lp(z, "theta_r", p)
        theta_s = _lp(z, "theta_s", p)
        theta_d = theta_r + (theta_s - theta_r) * torch.sigmoid(raw_th)

        return psi, u, w, theta_d


# ── Collocation points ────────────────────────────────────────
def sample_pts(n_int, n_bc, n_ic):
    def rn(n): return torch.rand(n, 1, device=device)
    def ze(n): return torch.zeros(n, 1, device=device)
    def on(n): return torch.ones(n, 1, device=device)
    def rq(*ts): return tuple(t.requires_grad_(True) for t in ts)
    return dict(
        int=rq(rn(n_int), rn(n_int), rn(n_int)),
        ic =rq(rn(n_ic),  rn(n_ic),  ze(n_ic)),
        top=rq(rn(n_bc),  on(n_bc),  rn(n_bc)),
        bot=rq(rn(n_bc),  ze(n_bc),  rn(n_bc)),
        lft=rq(ze(n_bc),  rn(n_bc),  rn(n_bc)),
        rgt=rq(on(n_bc),  rn(n_bc),  rn(n_bc)),
    )


def psi_initial(z_norm):
    return -(Z_MAX - z_norm * Z_MAX) * 0.4


_q_rain_fn = None


def q_rain_fallback(t_norm):
    return 1.94e-6 * torch.ones_like(t_norm)


def get_q_rain():
    return _q_rain_fn if _q_rain_fn is not None else q_rain_fallback


# ── Metrics ───────────────────────────────────────────────────
def compute_metrics(model, x_obs, z_obs, t_obs, theta_obs):
    """
    Returns R², RMSE, MAE and NSE all in one call.
    R² = coefficient of determination  (1.0 = perfect)
    RMSE = root mean squared error      (lower is better, in m³/m³)
    MAE  = mean absolute error          (lower is better, in m³/m³)
    NSE  = Nash-Sutcliffe Efficiency    (1.0 = perfect, >0.7 = acceptable)
    """
    with torch.no_grad():
        _, _, _, theta_d = model(x_obs, z_obs, t_obs)
        obs  = theta_obs
        pred = theta_d

        ss_res = torch.sum((pred - obs) ** 2)
        ss_tot = torch.sum((obs - obs.mean()) ** 2)

        r2   = (1.0 - ss_res / ss_tot).item() if ss_tot > 1e-12 else float("nan")
        rmse = torch.sqrt(torch.mean((pred - obs) ** 2)).item()
        mae  = torch.mean(torch.abs(pred - obs)).item()
        # NSE identical to R² for a single variable, listed separately for clarity
        nse  = r2

    return {"R2": r2, "RMSE": rmse, "MAE": mae, "NSE": nse}


def compute_r2(model, x_obs, z_obs, t_obs, theta_obs):
    """Backward-compatible single-value R² for the training loop."""
    return compute_metrics(model, x_obs, z_obs, t_obs, theta_obs)["R2"]


# ── PDE Losses ────────────────────────────────────────────────
def safe_mean_sq(residual, name=""):
    val = torch.mean(residual ** 2)
    if not torch.isfinite(val):
        print(f"  ⚠  NaN/Inf in '{name}' — skipping")
        return torch.tensor(0.0, device=residual.device, requires_grad=True)
    return val


def loss_richards(model, x, z, t):
    p   = model.hydro_params
    psi, _, _, _ = model(x, z, t)
    K   = vg_K(psi, z, p)
    C   = dtheta_dpsi(psi, z, p)
    Kx  = K * _grad(psi, x)
    Kz  = K * (_grad(psi, z) + 1.0)
    res = C * _grad(psi, t) - _grad(Kx, x) - _grad(Kz, z)
    return safe_mean_sq(res, "richards")


def loss_ic(model, x, z, t):
    psi, u, w, _ = model(x, z, t)
    return (safe_mean_sq(psi - psi_initial(z), "ic_psi")
          + safe_mean_sq(u,  "ic_u")
          + safe_mean_sq(w,  "ic_w"))


def loss_bc_top(model, x, z, t):
    p    = model.hydro_params
    psi, u, w, _ = model(x, z, t)
    K      = vg_K(psi, z, p)
    q_pred = -K * (_grad(psi, z) + 1.0)
    l_flux = safe_mean_sq(q_pred - get_q_rain()(t), "bc_top_flux")
    _, szz, txz = stress_2d(u, w, x, z)
    return l_flux + 0.05 * (safe_mean_sq(szz, "szz") + safe_mean_sq(txz, "txz"))


def loss_bc_bot(model, x, z, t):
    psi, u, w, _ = model(x, z, t)
    return (safe_mean_sq(psi, "bc_bot_psi")
          + safe_mean_sq(u,   "bc_bot_u")
          + safe_mean_sq(w,   "bc_bot_w"))


def loss_bc_sides(model, xl, zl, tl, xr, zr, tr):
    pl, _, _, _ = model(xl, zl, tl)
    pr, _, _, _ = model(xr, zr, tr)
    return (safe_mean_sq(_grad(pl, xl), "side_l")
          + safe_mean_sq(_grad(pr, xr), "side_r"))


def loss_mech(model, x, z, t):
    _, u, w, _ = model(x, z, t)
    sxx, szz, txz = stress_2d(u, w, x, z)
    body = _lp(z, "rho_b") * G_ACC / E_REF
    return (safe_mean_sq(_grad(sxx, x) + _grad(txz, z),         "mech_x")
          + safe_mean_sq(_grad(txz, x) + _grad(szz, z) + body,  "mech_z"))


def loss_data(model, x_obs, z_obs, t_obs, theta_obs):
    p   = model.hydro_params
    _, _, _, theta_d = model(x_obs, z_obs, t_obs)
    dts = _lp(z_obs, "theta_s", p) - _lp(z_obs, "theta_r", p)
    normalised_res = (theta_d - theta_obs) / (dts + 1e-6)
    return safe_mean_sq(normalised_res, "data_theta_norm")


def loss_consistency(model, x, z, t):
    p   = model.hydro_params
    psi, _, _, theta_d = model(x, z, t)
    psi_target = inv_vg_psi(theta_d.detach(), z, p)
    psi_scale  = torch.where(z * Z_MAX >= 8.0,  3.0 * torch.ones_like(z),
                 torch.where(z * Z_MAX >= 4.0, 10.0 * torch.ones_like(z),
                                               50.0 * torch.ones_like(z)))
    res = (psi - psi_target) / (psi_scale + 1e-6)
    return safe_mean_sq(res, "consistency")


def synthetic_sensors(n=300):
    xo = torch.rand(n, 1, device=device)
    zo = torch.rand(n, 1, device=device)
    to = torch.rand(n, 1, device=device)
    tr = _lp(zo, "theta_r"); ts = _lp(zo, "theta_s")
    t_h = to * T_MAX
    sat = 0.3 + 0.5 * torch.exp(-0.5 * ((t_h - T_MAX * 0.375) / (T_MAX * 0.1)) ** 2)
    th  = torch.clamp(tr + sat * (ts - tr) + 0.005 * torch.randn_like(tr), min=tr, max=ts)
    return xo, zo, to, th


# ── Two-Phase Training ────────────────────────────────────────
def train(warmup_epochs=500, total_epochs=18000, lr=2e-4,
          n_int=100, n_bc=150, n_ic=200,
          lam_h=1.0, lam_m=0.001, lam_ic=0.5,
          lam_bc=2.0, lam_data=60.0, lam_cons=5.0,
          csv_loader=None, print_every=250):

    global _q_rain_fn
    model = PINN(hidden=6, width=128).to(device)

    # ── Optimizer: ALL model.parameters() now includes hydraulic params ──
    # Hydraulic params use lower LR for stable inverse identification
    net_params   = list(model.trunk.parameters()) + \
                   list(model.head_psi.parameters()) + \
                   list(model.head_disp.parameters()) + \
                   list(model.head_theta.parameters())
    hydro_params = [model._log_alpha, model._log_Ks,
                    model._raw_n, model._raw_theta_r, model._raw_theta_s]

    opt = torch.optim.Adam([
        {"params": net_params,   "lr": lr,        "weight_decay": 1e-5},
        {"params": hydro_params, "lr": lr * 0.05, "weight_decay": 0.0},
        #                               ↑ 5% of net LR — slow, stable param updates
    ])
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
                opt, T_0=500, T_mult=1, eta_min=5e-7)

    use_csv = csv_loader is not None and getattr(csv_loader, "_loaded", False)
    if use_csv:
        csv_loader.to(device)
        xo  = csv_loader.x_obs_norm
        zo  = csv_loader.z_obs_norm
        to_ = csv_loader.t_obs_norm
        tho = csv_loader.theta_obs
        _q_rain_fn = csv_loader.make_q_rain_fn()
        print(f"  [Data]  CSV — {len(tho)} obs  T_MAX={T_MAX:.0f}h")
        print(f"  [z_obs] {(zo*Z_MAX).min().item():.2f}–{(zo*Z_MAX).max().item():.2f}m")
        print(f"  [theta] {tho.min().item():.3f}–{tho.max().item():.3f}  "
              f"mean={tho.mean().item():.3f}  std={tho.std().item():.4f}")
    else:
        xo, zo, to_, tho = synthetic_sensors(300)
        _q_rain_fn = None
        print("  [Data]  Synthetic sensors (no CSV)")

    init_m = compute_metrics(model, xo, zo, to_, tho)
    print(f"  [Init]  R²={init_m['R2']:.4f}  RMSE={init_m['RMSE']:.5f}  MAE={init_m['MAE']:.5f}")
    print(f"  [Phase] 1 = data+IC warmup (ep 1–{warmup_epochs})")
    print(f"  [Phase] 2 = all losses     (ep {warmup_epochs+1}–{total_epochs})")

    hist = defaultdict(list)
    param_hist = defaultdict(list)   # tracks inverse param evolution
    N   = len(tho)
    bar = "─" * 120
    print(f"\n{bar}")
    print(f"{'Ep':>6} {'Ph':>3} {'Total':>10} {'Data':>10} "
          f"{'Cons':>9} {'Hydro':>9} {'IC':>9} {'BC':>9} "
          f"{'R²':>8} {'RMSE':>9} {'MAE':>9}")
    print(bar)

    nan_count = 0
    for ep in range(1, total_epochs + 1):
        opt.zero_grad()
        phase = 1 if ep <= warmup_epochs else 2

        p_pts = sample_pts(n_int, n_bc, n_ic)
        xi, zi, ti = p_pts["int"]
        xc, zc, tc = p_pts["ic"]
        xt, zt, tt = p_pts["top"]
        xb, zb, tb = p_pts["bot"]
        xl, zl, tl = p_pts["lft"]
        xr, zr, tr = p_pts["rgt"]

        idx  = torch.randperm(N, device=device)[:min(N, 512)]
        xob  = xo[idx].requires_grad_(False)
        zob  = zo[idx].requires_grad_(False)
        tob  = to_[idx].requires_grad_(False)
        thob = tho[idx]

        lD    = loss_data(model, xob, zob, tob, thob)
        lCons = loss_consistency(model, xi, zi, ti)
        lI    = loss_ic(model, xc, zc, tc)

        if phase == 1:
            total = lam_data * lD + lam_cons * lCons + 0.3 * lI
            lH = torch.tensor(0.0); lB = torch.tensor(0.0)
        else:
            lH = loss_richards(model, xi, zi, ti)
            lB = (loss_bc_top  (model, xt, zt, tt)
                + loss_bc_bot  (model, xb, zb, tb)
                + loss_bc_sides(model, xl, zl, tl, xr, zr, tr))
            lM = loss_mech(model, xi, zi, ti)
            total = (lam_data * lD + lam_cons * lCons
                   + lam_ic * lI   + lam_bc * lB
                   + lam_h * lH    + lam_m * lM)

        if not torch.isfinite(total):
            nan_count += 1
            print(f"  ⚠  Ep {ep}: NaN total")
            if nan_count >= 5: print("  ✗  Stopping."); break
            sched.step(); continue

        nan_count = 0
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        model.clamp_params_()   # ← hard-clamp hydraulic params after every step
        sched.step()

        for k, v in [("total", total.item()), ("D", lD.item()),
                     ("Cons", lCons.item()),
                     ("H", lH if isinstance(lH, float) else lH.item()),
                     ("I", lI.item()),
                     ("B", lB if isinstance(lB, float) else lB.item())]:
            hist[k].append(v)

        # Track inverse param evolution every 50 epochs
        if ep % 50 == 0:
            lp = model.get_learned_params()
            for k, v in lp.items():
                param_hist[k].append((ep, v.copy()))

        if ep % print_every == 0 or ep == 1:
            m = compute_metrics(model, xo, zo, to_, tho)
            hist["r2"].append((ep, m["R2"]))
            hist["rmse"].append((ep, m["RMSE"]))
            hist["mae"].append((ep, m["MAE"]))
            lH_v = lH if isinstance(lH, float) else lH.item()
            lB_v = lB if isinstance(lB, float) else lB.item()
            print(f"{ep:>6d} Ph{phase}  {total.item():>10.3e}  {lD.item():>10.3e}  "
                  f"{lCons.item():>9.3e}  {lH_v:>9.3e}  {lI.item():>9.3e}  "
                  f"{lB_v:>9.3e}  "
                  f"{m['R2']:>8.4f}  {m['RMSE']:>9.5f}  {m['MAE']:>9.5f}")

    # ── Final metrics & learned parameters ────────────────────
    final_m = compute_metrics(model, xo, zo, to_, tho)
    lp_final = model.get_learned_params()

    print(f"{bar}")
    print(f"✓  Final  R²={final_m['R2']:.4f}  RMSE={final_m['RMSE']:.5f}  "
          f"MAE={final_m['MAE']:.5f}  NSE={final_m['NSE']:.4f}")
    _print_r2_advice(final_m["R2"])

    print("\n[Inverse Results] Learned vs Prior hydraulic parameters:")
    print(f"{'Param':<10} {'Layer':<12} {'Prior':>12} {'Learned':>12} {'Δ%':>8}")
    print("─" * 56)
    for key in ["alpha", "n", "Ks", "theta_r", "theta_s"]:
        for li, layer_name in enumerate(["Sandy CL", "Clay Loam", "Clay"]):
            prior   = SOIL[li+1][key]
            learned = lp_final[key][li]
            delta   = 100.0 * (learned - prior) / (abs(prior) + 1e-20)
            print(f"  {key:<8} {layer_name:<12} {prior:>12.4e}  {learned:>12.4e}  {delta:>+7.1f}%")
    print("─" * 56)

    hist["param_hist"] = param_hist
    return model, hist, final_m


def _print_r2_advice(r2):
    if r2 >= 0.96:   print("  ✓  Excellent — R² achieved!")
    elif r2 >= 0.85: print("  ✓  Good fit. Try lam_data=60 or more epochs.")
    elif r2 >= 0.60: print("  ⚠  Moderate. Raise lam_data to 60–80, epochs to 8000.")
    else:            print("  ✗  Poor fit. Run plot_parity() to diagnose.")


# ── Factor of Safety (uses trainable params) ──────────────────
def compute_fos(model, t_norm=1.0, n=50):
    a  = SLOPE_ANGLE * np.pi / 180.0
    xv = torch.linspace(0, 1, n, device=device)
    zv = torch.linspace(0, 1, n, device=device)
    Xg, Zg = torch.meshgrid(xv, zv, indexing="ij")
    Xf, Zf = Xg.reshape(-1, 1), Zg.reshape(-1, 1)
    Tf = torch.full_like(Xf, t_norm)
    with torch.no_grad():
        psi, _, _, _ = model(Xf, Zf, Tf)
    z_m   = Zf * Z_MAX
    rho_b = _lp(Zf, "rho_b"); c_p = _lp(Zf, "c_prime")
    phi_p = _lp(Zf, "phi_prime") * np.pi / 180.0
    H     = Z_MAX - z_m
    u_w   = torch.clamp(RHO_W * G_ACC * psi, min=-1e5, max=1e5)
    sigma_n = torch.clamp(rho_b * G_ACC * H * (np.cos(a) ** 2) - u_w, min=0.0)
    tau   = rho_b * G_ACC * H * np.sin(a) * np.cos(a)
    tau_f = c_p + sigma_n * torch.tan(phi_p)
    fos   = tau_f / (torch.abs(tau) + 1e-6)
    return fos.reshape(n, n).cpu().numpy(), Xg.cpu().numpy() * X_MAX, Zg.cpu().numpy() * Z_MAX


# ── Plots ─────────────────────────────────────────────────────
LAYER_LINES  = [4.0, 8.0]
LAYER_LABELS = {9.0: "Sandy Clay Loam", 6.0: "Clay Loam", 2.0: "Clay"}


def _add_layers(ax):
    for zl in LAYER_LINES:
        ax.axhline(zl, color="k", ls="--", lw=0.8, alpha=0.6)


def plot_metrics(hist, save="metrics_v5_inverse.png"):
    """
    Plots R², RMSE, MAE over training epochs + all loss curves.
    This is your key accuracy figure for the paper.
    """
    r2_data   = hist.get("r2",   [])
    rmse_data = hist.get("rmse", [])
    mae_data  = hist.get("mae",  [])

    fig = plt.figure(figsize=(20, 14))
    gs  = fig.add_gridspec(4, 3, hspace=0.45, wspace=0.35)

    # Row 0–1: individual loss curves
    keys   = ["total", "D", "Cons", "H", "I", "B"]
    labels = ["Total", "Data (norm)", "Consistency", "Hydro (Richards)", "IC", "BC"]
    colors = ["black", "purple", "teal", "steelblue", "seagreen", "darkorange"]
    for idx, (k, lbl, col) in enumerate(zip(keys, labels, colors)):
        r, c = divmod(idx, 3)
        ax   = fig.add_subplot(gs[r, c])
        d    = hist.get(k, [])
        if not d: continue
        ax.semilogy(d, color=col, lw=1.5)
        ax.set_title(lbl, fontweight="bold", fontsize=10)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.grid(alpha=0.3)

    # Row 2: R² evolution
    ax_r2 = fig.add_subplot(gs[2, :])
    if r2_data:
        eps, r2s = zip(*r2_data)
        ax_r2.plot(eps, r2s, "o-", color="darkgreen", lw=2, ms=5, label="R²")
        ax_r2.axhline(0.96, color="green", ls="--", lw=1.2, alpha=0.8, label="R²=0.96 target")
        ax_r2.axhline(0.70, color="orange", ls="--", lw=1.0, alpha=0.7, label="NSE=0.70 acceptable")
        ax_r2.set_ylim(-0.3, 1.08)
        ax_r2.set_xlabel("Epoch"); ax_r2.set_ylabel("R² / NSE")
        ax_r2.set_title("R² (= NSE) Evolution — Model Accuracy", fontweight="bold")
        ax_r2.legend(fontsize=9); ax_r2.grid(alpha=0.3)
        if r2s:
            ax_r2.text(eps[-1], min(r2s[-1] + 0.04, 1.02),
                       f"R²={r2s[-1]:.4f}", ha="right",
                       fontsize=12, color="darkgreen", fontweight="bold")

    # Row 3: RMSE and MAE side by side
    ax_rmse = fig.add_subplot(gs[3, :2])
    ax_mae  = fig.add_subplot(gs[3, 2])

    if rmse_data:
        eps_r, rmse_vals = zip(*rmse_data)
        ax_rmse.plot(eps_r, rmse_vals, "s-", color="firebrick", lw=2, ms=5, label="RMSE")
        if mae_data:
            eps_m, mae_vals = zip(*mae_data)
            ax_rmse.plot(eps_m, mae_vals, "^-", color="darkorange", lw=2, ms=5, label="MAE")
        ax_rmse.set_xlabel("Epoch"); ax_rmse.set_ylabel("Error (m³/m³)")
        ax_rmse.set_title("RMSE & MAE Evolution", fontweight="bold")
        ax_rmse.legend(fontsize=9); ax_rmse.grid(alpha=0.3)
        if rmse_vals:
            ax_rmse.text(eps_r[-1], rmse_vals[-1],
                         f"  RMSE={rmse_vals[-1]:.5f}", fontsize=10,
                         color="firebrick", fontweight="bold")

    if mae_data:
        eps_m, mae_vals = zip(*mae_data)
        ax_mae.plot(eps_m, mae_vals, "^-", color="darkorange", lw=2, ms=5)
        ax_mae.set_xlabel("Epoch"); ax_mae.set_ylabel("MAE (m³/m³)")
        ax_mae.set_title(f"Final MAE = {mae_vals[-1]:.5f}", fontweight="bold")
        ax_mae.grid(alpha=0.3)

    fig.suptitle("PINN v5 Inverse — Training History & Accuracy Metrics",
                 fontsize=14, fontweight="bold")
    plt.savefig(save, dpi=150, bbox_inches="tight")
    print(f"[Plot] Metrics → {save}")
    plt.show()


def plot_inverse_params(hist, save="inverse_params_v5.png"):
    """
    Plots the convergence of each learned hydraulic parameter over training.
    Essential figure for convincing reviewers about parameter identification.
    """
    param_hist = hist.get("param_hist", {})
    if not param_hist:
        print("No parameter history to plot."); return

    layer_names = ["Sandy CL (L1)", "Clay Loam (L2)", "Clay (L3)"]
    layer_colors = ["steelblue", "seagreen", "darkorange"]
    param_keys  = ["alpha", "n", "Ks", "theta_r", "theta_s"]
    param_units = ["1/m", "—", "m/s", "m³/m³", "m³/m³"]

    fig, axes = plt.subplots(len(param_keys), 1, figsize=(14, 16))
    fig.suptitle("Inverse Modeling — Hydraulic Parameter Convergence\n"
                 "(dashed = PTF prior, solid = learned)",
                 fontsize=13, fontweight="bold")

    for ax, key, unit in zip(axes, param_keys, param_units):
        entries = param_hist.get(key, [])
        if not entries:
            continue
        eps_arr = [e[0] for e in entries]
        vals    = np.array([e[1] for e in entries])   # shape [T, 3]

        for li in range(3):
            prior = SOIL[li+1][key]
            ax.axhline(prior, color=layer_colors[li], ls="--", lw=1.5,
                       alpha=0.5, label=f"{layer_names[li]} prior={prior:.4g}")
            ax.plot(eps_arr, vals[:, li], color=layer_colors[li],
                    lw=2.0, label=f"{layer_names[li]} learned")

        ax.set_ylabel(f"{key} ({unit})", fontsize=10)
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7, loc="right")

        if key == "Ks":
            ax.set_yscale("log")
            ax.set_title(f"{key} — log scale (prior=PTF, learned=inverse PINN)",
                         fontsize=9, fontweight="bold")
        else:
            ax.set_title(f"{key} — convergence", fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.savefig(save, dpi=150, bbox_inches="tight")
    print(f"[Plot] Inverse params → {save}")
    plt.show()


def plot_parity(model, x_obs, z_obs, t_obs, theta_obs, save="parity_v5.png"):
    """
    Parity plot + time series + residual distribution.
    Reports R², RMSE, MAE, NSE.
    """
    with torch.no_grad():
        _, _, _, theta_d = model(x_obs, z_obs, t_obs)
        th_hat = theta_d.cpu().numpy().flatten()
    obs = theta_obs.cpu().numpy().flatten()
    t_v = t_obs.cpu().numpy().flatten() * T_MAX

    m    = compute_metrics(model, x_obs, z_obs, t_obs, theta_obs)
    r2   = m["R2"]; rmse = m["RMSE"]; mae = m["MAE"]; nse = m["NSE"]
    res  = th_hat - obs

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(
        f"Model Accuracy — R²={r2:.4f}  |  RMSE={rmse:.5f} m³/m³  |  "
        f"MAE={mae:.5f} m³/m³  |  NSE={nse:.4f}",
        fontsize=13, fontweight="bold")

    # Parity plot
    ax = axes[0]
    sc = ax.scatter(obs, th_hat, s=3, alpha=0.35, c=t_v / T_MAX, cmap="plasma")
    plt.colorbar(sc, ax=ax, label="t (normalised)")
    mn = min(obs.min(), th_hat.min()); mx = max(obs.max(), th_hat.max())
    ax.plot([mn, mx], [mn, mx], "r--", lw=1.5, label="1:1 line")
    ax.set_xlabel("Observed θ (m³/m³)"); ax.set_ylabel("Predicted θ (m³/m³)")
    ax.set_title("Parity Plot", fontweight="bold"); ax.legend(); ax.grid(alpha=0.3)

    # Time series
    ax = axes[1]
    sort_idx = np.argsort(t_v)
    ax.plot(t_v[sort_idx], obs[sort_idx],     "b-",  lw=0.7, alpha=0.7, label="Observed")
    ax.scatter(t_v[sort_idx], th_hat[sort_idx], s=2, c="red", alpha=0.4, label="Predicted")
    ax.set_xlabel("Time (h)"); ax.set_ylabel("θ (m³/m³)")
    ax.set_title(f"θ Time Series  (R²={r2:.4f})", fontweight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Residual distribution
    ax = axes[2]
    ax.hist(res, bins=60, color="steelblue", ec="k", lw=0.3, alpha=0.85)
    ax.axvline(0,        color="red",    ls="--", lw=1.5, label="Zero bias")
    ax.axvline(res.mean(), color="orange", ls="-",  lw=1.5,
               label=f"Mean={res.mean():.5f}")
    ax.set_xlabel("Residual θ_pred − θ_obs (m³/m³)")
    ax.set_ylabel("Count")
    ax.set_title(f"Residual Distribution  (RMSE={rmse:.5f})", fontweight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save, dpi=150, bbox_inches="tight")
    print(f"[Plot] Parity → {save}  R²={r2:.4f}  RMSE={rmse:.5f}  MAE={mae:.5f}")
    plt.show()
    return r2, rmse, mae


def plot_loss(hist, save="loss_v5.png"):
    """Legacy loss plot — kept for compatibility."""
    plot_metrics(hist, save=save)


def plot_fields(model, t_norm=0.5, save="fields_v5.png"):
    n = 60
    xv = torch.linspace(0, 1, n, device=device)
    zv = torch.linspace(0, 1, n, device=device)
    Xg, Zg = torch.meshgrid(xv, zv, indexing="ij")
    Xf, Zf = Xg.reshape(-1, 1), Zg.reshape(-1, 1)
    Tf = torch.full_like(Xf, t_norm)
    p  = model.hydro_params
    with torch.no_grad():
        psi, u, w, theta_d = model(Xf, Zf, Tf)
        K = vg_K(psi, Zf, p)
    def r(a): return a.cpu().numpy().reshape(n, n)
    Xn, Zn = Xg.cpu().numpy() * X_MAX, Zg.cpu().numpy() * Z_MAX
    panels  = [(r(theta_d), "θ direct (m³/m³)", "Blues"),
               (r(psi),     "ψ (m)",             "RdBu_r"),
               (np.log10(r(K) + 1e-16), "log₁₀ K (m/s)", "YlGn"),
               (r(u) * 1000, "u (mm)", "PuOr"),
               (r(w) * 1000, "w (mm)", "RdYlBu"),
               (RHO_W * G_ACC * r(psi) / 1000, "u_w (kPa)", "hot_r")]
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(f"PINN v5 Inverse Fields — t={t_norm*T_MAX:.0f}h",
                 fontsize=14, fontweight="bold")
    for ax, (arr, lbl, cm) in zip(axes.flat, panels):
        cf = ax.contourf(Xn, Zn, arr, levels=20, cmap=cm)
        plt.colorbar(cf, ax=ax, shrink=0.82)
        ax.set_title(lbl, fontsize=10, fontweight="bold")
        ax.set_xlabel("x (m)"); ax.set_ylabel("z (m)")
        _add_layers(ax)
    plt.tight_layout()
    plt.savefig(save, dpi=150, bbox_inches="tight")
    print(f"[Plot] Fields → {save}"); plt.show()


def plot_fos_map(model, t_norm=1.0, save="fos_v5.png"):
    fos, Xn, Zn = compute_fos(model, t_norm, n=60)
    fc = np.clip(fos, 0, 5)
    fig, ax = plt.subplots(figsize=(10, 6))
    cf = ax.contourf(Xn, Zn, fc, levels=25, cmap="RdYlGn", vmin=0.5, vmax=4.0)
    plt.colorbar(cf, ax=ax, label="FoS")
    cs = ax.contour(Xn, Zn, fc, levels=[1.0], colors="red", linewidths=2)
    ax.clabel(cs, fmt="FoS=1.0", fontsize=9, colors="red")
    _add_layers(ax)
    for zm, ln in LAYER_LABELS.items():
        ax.text(X_MAX * 0.01, zm, ln, va="center", ha="left", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))
    ax.set(xlabel="x (m)", ylabel="z (m)",
           title=f"FoS — t={t_norm*T_MAX:.0f}h  (slope={SLOPE_ANGLE}°)")
    plt.tight_layout()
    plt.savefig(save, dpi=150, bbox_inches="tight")
    print(f"[Plot] FoS → {save}"); plt.show()
    return fos


def plot_fos_evolution(model, save="fos_evo_v5.png"):
    snaps = [0.0, 0.25, 0.5, 0.75, 1.0]
    fig, axes = plt.subplots(1, 5, figsize=(22, 5), sharey=True)
    fig.suptitle("FoS Evolution Over Time", fontsize=13, fontweight="bold")
    track = []
    for ax, tn in zip(axes, snaps):
        fos, Xn, Zn = compute_fos(model, tn, n=40)
        fc = np.clip(fos, 0, 5); track.append(fc.min())
        cf = ax.contourf(Xn, Zn, fc, levels=15, cmap="RdYlGn", vmin=0.5, vmax=4.0)
        ax.contour(Xn, Zn, fc, levels=[1.0], colors="red", linewidths=1.5)
        _add_layers(ax)
        ax.set_title(f"t={tn*T_MAX:.0f}h\nFoS_min={fc.min():.2f}",
                     fontsize=9, fontweight="bold")
        ax.set_xlabel("x (m)", fontsize=9)
        if ax is axes[0]: ax.set_ylabel("z (m)", fontsize=9)
        plt.colorbar(cf, ax=ax, shrink=0.75)
    plt.tight_layout()
    plt.savefig(save, dpi=150, bbox_inches="tight")
    print(f"[Plot] FoS evolution → {save}"); plt.show()
    return track


# ── CSV Loader (unchanged from v5) ───────────────────────────
class CSVDataLoader:
    def __init__(self, filepath=None):
        self.filepath  = filepath
        self.df        = None
        self._loaded   = False
        self.depth_val = None
        for a in ["t_rain_h", "q_rain_ms", "x_obs_norm", "z_obs_norm",
                  "t_obs_norm", "theta_obs", "x_ic_norm", "z_ic_norm", "theta_ic"]:
            setattr(self, a, None)

    def load(self):
        if not self.filepath or not os.path.exists(self.filepath):
            raise FileNotFoundError(f"'{self.filepath}' not found.")
        print(f"\n{'─'*62}\n[CSV] {self.filepath}\n{'─'*62}")
        df = self._read()
        df = self._parse_time(df)
        df = self._parse_geo(df)
        df = self._parse_depth(df)
        df = self._parse_rain(df)
        df = self._parse_soil(df)
        self.df = df
        self._build(df)
        self._summary(df)
        self._loaded = True
        return self

    def _read(self):
        for sep in [",", ";", "\t", "|"]:
            try:
                df = pd.read_csv(self.filepath, sep=sep,
                                 skipinitialspace=True, encoding="utf-8-sig")
                if df.shape[1] >= 4:
                    df.columns = [c.strip().lower() for c in df.columns]
                    print(f"  Rows:{len(df)}  Cols:{list(df.columns)}  sep='{sep}'")
                    return df
            except Exception:
                continue
        raise ValueError("Cannot parse CSV.")

    def _col(self, df, u):
        lo = u.lower()
        if lo in df.columns: return lo
        m = [c for c in df.columns if lo in c or c in lo]
        if m: print(f"  [Fuzzy] '{u}' → '{m[0]}'"); return m[0]
        raise KeyError(f"Column '{u}' not found. Have: {list(df.columns)}")

    def _parse_time(self, df):
        col = self._col(df, COL_TIME)
        raw = df[col].astype(str).str.strip()
        num = pd.to_numeric(raw, errors="coerce")
        if num.notna().all():
            vals    = num.values.astype(float)
            elapsed = vals - vals.min() if vals.mean() < 1e6 else (vals - vals.min()) / 3600.0
            df["t_h"] = elapsed
            self._update_tmax(elapsed.max())
            return df
        parsed, used = None, None
        for fmt in TIMESTAMP_FORMATS:
            try:
                s = pd.to_datetime(raw, format=fmt, errors="raise")
                parsed, used = s, fmt; break
            except Exception:
                pass
        if parsed is None:
            parsed = pd.to_datetime(raw, dayfirst=False, errors="coerce")
            used   = "inferred"
        df = df[parsed.notna()].copy()
        parsed = parsed[parsed.notna()]
        elapsed = (parsed - parsed.min()).dt.total_seconds().values / 3600.0
        df["t_h"] = elapsed
        print(f"  Time : '{used}'  [{elapsed.min():.1f}, {elapsed.max():.1f}]h")
        self._update_tmax(elapsed.max())
        return df

    def _update_tmax(self, v):
        global T_MAX
        if v > T_MAX:
            T_MAX = float(np.ceil(v / 6.0) * 6.0)
            print(f"  T_MAX → {T_MAX:.0f}h")

    def _parse_geo(self, df):
        global X_MAX
        col = self._col(df, COL_GEO)
        x   = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
        x_m = (x.values - x.values.min()) * (111320.0 if GEO_IS_LATLON else 1.0)
        if x_m.max() < 0.01:
            print(f"  Geo  : constant → x assigned to mid-domain ({X_MAX/2:.1f}m)")
            x_m = np.full_like(x_m, X_MAX / 2.0)
        else:
            if x_m.max() > X_MAX:
                X_MAX = float(np.ceil(x_m.max() / 5.0) * 5.0)
                print(f"  X_MAX → {X_MAX:.0f}m")
        df["x_m"] = x_m
        return df

    def _parse_depth(self, df):
        col       = self._col(df, COL_DEPTH)
        depth_raw = pd.to_numeric(df[col], errors="coerce").ffill().bfill()
        self.depth_val = depth_raw.iloc[0]
        z_m = (Z_MAX - depth_raw.values) if DEPTH_POS_DOWN else depth_raw.values
        df["z_m"] = np.clip(z_m, 0.0, Z_MAX)
        layer = ("Sandy CL" if df["z_m"].iloc[0] >= 8 else
                 "Clay Loam" if df["z_m"].iloc[0] >= 4 else "Clay")
        print(f"  Depth: '{col}'={self.depth_val:.0f}m → z={df['z_m'].iloc[0]:.1f}m ({layer})")
        return df

    def _parse_rain(self, df):
        col = self._col(df, COL_RAINFALL)
        f   = {"mm/hr": 1/3.6e6, "mm/day": 1/86400000.0, "mm/s": 0.001,
               "m/s": 1.0, "m/hr": 1/3600.0}.get(RAINFALL_UNIT.lower(), 1/3.6e6)
        r   = pd.to_numeric(df[col], errors="coerce").fillna(0.0).clip(lower=0)
        df["q_ms"] = r.values * f
        return df

    def _parse_soil(self, df):
        col = self._col(df, COL_SOIL)
        s   = pd.to_numeric(df[col], errors="coerce").ffill().bfill()
        if s.max() > 1.0:
            s = s / 100.0
        s = s.clip(THETA_CLIP_LO, THETA_CLIP_HI)
        df["theta"] = s.values
        print(f"  Soil : θ=[{df['theta'].min():.3f}, {df['theta'].max():.3f}]  "
              f"mean={df['theta'].mean():.3f}")
        return df

    def _build(self, df):
        df = df.copy()
        df["x_n"] = (df["x_m"] / X_MAX).clip(0, 1)
        df["z_n"] = (df["z_m"] / Z_MAX).clip(0, 1)
        df["t_n"] = (df["t_h"] / T_MAX).clip(0, 1)

        def _t(c):   return torch.tensor(df[c].values,  dtype=torch.float32).unsqueeze(1)
        def _tic(c): return torch.tensor(ic[c].values,  dtype=torch.float32).unsqueeze(1)

        rain = (df.groupby("t_h")["q_ms"].max()
                  .reset_index().sort_values("t_h"))
        self.t_rain_h  = torch.tensor(rain["t_h"].values,  dtype=torch.float32)
        self.q_rain_ms = torch.tensor(rain["q_ms"].values, dtype=torch.float32)
        self.x_obs_norm = _t("x_n"); self.z_obs_norm = _t("z_n")
        self.t_obs_norm = _t("t_n"); self.theta_obs  = _t("theta")
        t0 = df["t_h"].min()
        ic = df[df["t_h"] == t0]
        if len(ic) < 3:
            ic = df.nsmallest(max(5, len(df) // 20), "t_h")
        self.x_ic_norm = _tic("x_n"); self.z_ic_norm = _tic("z_n")
        self.theta_ic  = _tic("theta")
        print(f"  Built: rain={len(self.t_rain_h)}  θ_obs={len(self.theta_obs)}")

    def _summary(self, df):
        depth_from_base = df["z_m"].iloc[0]
        layer = ("Sandy CL" if depth_from_base >= 8 else
                 "Clay Loam" if depth_from_base >= 4 else "Clay")
        print(f"\n  ┌─────────────────────────────────────────────────────┐")
        print(f"  │ Rows         : {len(df):<37}│")
        print(f"  │ Time         : {df['t_h'].min():.1f} – {df['t_h'].max():.1f} h{'':>28}│")
        print(f"  │ Sensor depth : {self.depth_val:.0f}m from surf → z={depth_from_base:.1f}m ({layer}){'':>5}│")
        print(f"  │ θ            : {df['theta'].min():.3f} – {df['theta'].max():.3f}  mean={df['theta'].mean():.3f}{'':>17}│")
        print(f"  └─────────────────────────────────────────────────────┘")

    def make_q_rain_fn(self):
        t_arr = (self.t_rain_h.numpy() / T_MAX).astype(np.float64)
        q_arr = self.q_rain_ms.numpy().astype(np.float64)
        def fn(t_norm):
            t_np = t_norm.detach().cpu().numpy().flatten().astype(np.float64)
            q_np = np.interp(t_np, t_arr, q_arr, left=0.0, right=0.0)
            return torch.tensor(q_np, dtype=torch.float32,
                                device=t_norm.device).reshape_as(t_norm)
        return fn

    def to(self, dev):
        for a in ["t_rain_h", "q_rain_ms", "x_obs_norm", "z_obs_norm",
                  "t_obs_norm", "theta_obs", "x_ic_norm", "z_ic_norm", "theta_ic"]:
            v = getattr(self, a)
            if v is not None: setattr(self, a, v.to(dev))
        return self

    def plot_raw_data(self, save="csv_overview_v5.png"):
        df = self.df
        fig, axes = plt.subplots(2, 2, figsize=(15, 9))
        fig.suptitle("CSV Data Overview", fontsize=13, fontweight="bold")
        ax = axes[0, 0]
        ax.fill_between(df["t_h"], df["q_ms"] * 3.6e6, alpha=0.55, color="steelblue")
        ax.set(xlabel="Time (h)", ylabel="Rainfall (mm/hr)", title="Rainfall")
        ax.grid(alpha=0.3)
        ax = axes[0, 1]
        sc = ax.scatter(df["t_h"], df["theta"], c=df["t_h"]/df["t_h"].max(),
                        cmap="plasma", s=3, alpha=0.5)
        plt.colorbar(sc, ax=ax, label="t (norm)")
        ax.set(xlabel="Time (h)", ylabel="θ", title="Soil Moisture")
        ax.grid(alpha=0.3)
        ax = axes[1, 0]; ax2 = ax.twinx()
        ax.plot(df["t_h"], df["theta"], color="saddlebrown", lw=0.7, alpha=0.8, label="θ")
        ax2.fill_between(df["t_h"], df["q_ms"]*3.6e6, alpha=0.25, color="steelblue", label="Rain")
        ax.set(xlabel="Time (h)", ylabel="θ", title="θ & Rain")
        ax2.set_ylabel("Rain (mm/hr)"); ax.grid(alpha=0.3)
        ax = axes[1, 1]
        ax.hist(df["theta"], bins=60, color="darkorange", ec="k", lw=0.3, alpha=0.85)
        ax.set(xlabel="θ", ylabel="Count", title="θ Distribution"); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(save, dpi=150, bbox_inches="tight")
        print(f"[Plot] CSV overview → {save}"); plt.show()


# ── Main ──────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 65)
    print("  PINN v5 INVERSE — trainable hydraulic parameters")
    print("=" * 65)

    loader = None
    if CSV_FILE and os.path.exists(CSV_FILE):
        loader = CSVDataLoader(CSV_FILE)
        try:
            loader.load()
            loader.plot_raw_data()
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f"  ⚠  CSV load failed: {e}"); loader = None

    model, hist, final_metrics = train(
        warmup_epochs = 500,
        total_epochs  = 18000,
        lr            = 2e-4,
        n_int=100, n_bc=150, n_ic=200,
        lam_h    = 1.0,
        lam_m    = 0.001,
        lam_ic   = 0.5,
        lam_bc   = 2.0,
        lam_data = 60.0,
        lam_cons = 5.0,
        csv_loader  = loader,
        print_every = 250,
    )

    torch.save(model.state_dict(), "pinn_v5_inverse.pth")
    print("[Saved] pinn_v5_inverse.pth")

    # ── Accuracy & inverse parameter plots ────────────────────
    plot_metrics(hist)
    plot_inverse_params(hist)

    if loader and loader._loaded:
        loader.to(device)
        r2_f, rmse_f, mae_f = plot_parity(
            model, loader.x_obs_norm, loader.z_obs_norm,
            loader.t_obs_norm, loader.theta_obs)
    else:
        xo, zo, to_, tho = synthetic_sensors(300)
        r2_f, rmse_f, mae_f = plot_parity(model, xo, zo, to_, tho)

    plot_fields(model, t_norm=0.5)
    fos_final = plot_fos_map(model, t_norm=1.0)
    track     = plot_fos_evolution(model)

    fc = np.clip(fos_final, 0, 10)
    print("\n" + "=" * 60)
    print("  v5 Inverse — Final Results Summary")
    print("=" * 60)
    print(f"  R²   (θ direct) : {r2_f:.4f}")
    print(f"  RMSE            : {rmse_f:.5f} m³/m³")
    print(f"  MAE             : {mae_f:.5f} m³/m³")
    print(f"  NSE             : {final_metrics['NSE']:.4f}")
    print(f"  Min  FoS        : {fc.min():.3f}")
    print(f"  Mean FoS        : {fc.mean():.3f}")
    print(f"  FoS < 1.0       : {(fc < 1.0).mean()*100:.1f}% of domain")
    print("  ⚠  FAILURE ZONE" if (fc < 1.0).any() else "  ✓  Slope stable")
    print("=" * 60)